# 04. Data Drift Monitoring (Evidently AI) & Production Serving
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheNarhawl/mind-dataset-recsys/blob/main/notebooks/04_drift_monitoring_and_export.ipynb)

This notebook covers:
1. Feature and interaction drift monitoring with **Evidently AI** (Reference vs Current temporal windows).
2. Generating and saving interactive HTML Data Drift reports.
3. Prototyping and profiling the end-to-end **FastAPI** TwoStageRecommender service (p95 SLA < 30 ms).


In [ ]:
# 1. Setup environment & repository in Google Colab
import sys
from pathlib import Path

# Clone repo if running in Google Colab
if not Path("src/mind_recsys").exists():
    !git clone https://github.com/TheNarhawl/mind-dataset-recsys.git
    %cd mind-dataset-recsys

# Add src to sys.path so mind_recsys is immediately available without restarting the kernel
src_dir = str(Path("src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Install required dependencies
!pip install -q pydantic pydantic-settings python-dotenv sentence-transformers qdrant-client catboost mlflow 'pandas<3.0.0' fastapi uvicorn evidently


In [ ]:
import numpy as np
import pandas as pd
from mind_recsys.monitoring.drift import DriftMonitor

# 1. Simulate Reference (Week 1) vs Current (Week 2) datasets
np.random.seed(42)
ref_features = pd.DataFrame({
    'cos_sim_user_item': np.random.normal(0.65, 0.15, 1000),
    'item_recency_hours': np.random.exponential(12.0, 1000),
    'item_global_ctr': np.random.beta(2, 20, 1000),
    'category_affinity': np.random.beta(3, 5, 1000),
})

# Week 2 experiences shift in recency and category affinity
curr_features = pd.DataFrame({
    'cos_sim_user_item': np.random.normal(0.63, 0.16, 1000),
    'item_recency_hours': np.random.exponential(24.0, 1000), # News aging shift
    'item_global_ctr': np.random.beta(2, 20, 1000),
    'category_affinity': np.random.beta(1.5, 6, 1000),     # User interest shift
})

monitor = DriftMonitor()
report_path = "./artifacts/drift_report.html"
report = monitor.generate_report(ref_features, curr_features, output_html_path=report_path)
summary = monitor.detect_drift_summary(ref_features, curr_features)
print(f"Data Drift Report saved to: {report_path}")
print("Drift Summary:", summary)


In [ ]:
# 2. Production Latency Profiling (100 sequential recommendations)
import time
from mind_recsys.retrieval.qdrant_index import QdrantNewsIndex
from mind_recsys.ranking.model import NewsRanker
from mind_recsys.service.pipeline import TwoStageRecommender
from mind_recsys.features.pipeline import FEATURE_NAMES

# Setup mock pipeline components
qdrant = QdrantNewsIndex(collection_name="serve_test", vector_size=384, location=":memory:")
qdrant.create_collection(recreate=True)
dummy_embs = np.random.randn(200, 384).astype(np.float32)
qdrant.index_news([f"N{i}" for i in range(200)], dummy_embs)

ranker = NewsRanker(iterations=10, depth=2)
ranker.fit(pd.DataFrame(np.random.randn(20, len(FEATURE_NAMES)), columns=FEATURE_NAMES), np.array([1, 0]*10), np.repeat(range(10), 2))

pipeline = TwoStageRecommender(
    retrieval_index=qdrant,
    ranker=ranker,
    candidate_pool_size=100
)

latencies = []
for _ in range(100):
    t0 = time.perf_counter()
    resp = pipeline.recommend(user_id="U_PROD", top_k=10)
    latencies.append((time.perf_counter() - t0) * 1000.0)

p50 = np.percentile(latencies, 50)
p95 = np.percentile(latencies, 95)
p99 = np.percentile(latencies, 99)

print(f"End-to-End Latency Benchmark (100 requests):")
print(f"  p50: {p50:.2f} ms")
print(f"  p95: {p95:.2f} ms (SLA < 30 ms: {'PASS' if p95 < 30 else 'FAIL'})")
print(f"  p99: {p99:.2f} ms")
